In [ ]:
!pip install -q transformers accelerate peft

In [ ]:
import os, random
from glob import glob
from typing import Dict, List, Tuple

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFile, UnidentifiedImageError

from transformers import CLIPProcessor, CLIPModel
from peft import LoraConfig, get_peft_model

# ---- PIL safety settings ----
Image.MAX_IMAGE_PIXELS = None          # disable decompression bomb limit
ImageFile.LOAD_TRUNCATED_IMAGES = True # allow truncated images

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
import os, random
from glob import glob
from typing import Dict, List, Tuple

# CHANGE this to your extracted "clean" dataset root
DATA_ROOT = "/content/drive/MyDrive/clean1/clean"

FOLDER_TO_LABEL = {
    "UML Activity":              "activity",
    "UML Class":                 "class",
    "UML Communication Diagram": "communication",
    "UML Component Diagram":     "component",
    "UML Deployment Diagram":    "deployment",
    "UML Object Diagram":        "object",
    "UML Package Diagram":       "package",
    "UML Sequence":              "sequence",
    "UML State Machine Diagram": "state_machine",
    "UML Use case":              "use_case",
    "UML non-UML":               "non_uml",
}

label_names = sorted(set(FOLDER_TO_LABEL.values()))
label_to_id = {lbl: i for i, lbl in enumerate(label_names)}
id_to_label = {i: lbl for lbl, i in label_to_id.items()}
num_labels = len(label_names)

print("Labels:", label_names)
print("num_labels:", num_labels)


def build_examples(root_dir: str, folder_to_label: Dict[str, str]) -> List[Tuple[str, int]]:
    """Collect image paths and label indices from folders."""
    examples = []
    for folder_name, label_name in folder_to_label.items():
        label_id = label_to_id[label_name]
        folder_path = os.path.join(root_dir, folder_name)
        if not os.path.isdir(folder_path):
            print("⚠️ folder missing:", folder_path)
            continue

        files = []
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
            files.extend(glob(os.path.join(folder_path, ext)))

        for f in files:
            examples.append((f, label_id))

    random.shuffle(examples)
    print(f"Total images found (raw): {len(examples)}")
    return examples


def split_examples(
    examples: List[Tuple[str, int]],
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
):
    random.seed(seed)
    n = len(examples)
    idx = list(range(n))
    random.shuffle(idx)

    n_train = int(train_ratio * n)
    n_val   = int(val_ratio * n)

    train_idx = idx[:n_train]
    val_idx   = idx[n_train:n_train + n_val]
    test_idx  = idx[n_train + n_val:]

    train_ex = [examples[i] for i in train_idx]
    val_ex   = [examples[i] for i in val_idx]
    test_ex  = [examples[i] for i in test_idx]

    print(f"Train: {len(train_ex)}, Val: {len(val_ex)}, Test: {len(test_ex)}")
    return train_ex, val_ex, test_ex


# ---- Build + split (no slow cleaning pass) ----
examples = build_examples(DATA_ROOT, FOLDER_TO_LABEL)
train_ex, val_ex, test_ex = split_examples(examples)


Labels: ['activity', 'class', 'communication', 'component', 'deployment', 'non_uml', 'object', 'package', 'sequence', 'state_machine', 'use_case']
num_labels: 11
Total images found (raw): 4725
Train: 3307, Val: 708, Test: 710


In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import CLIPProcessor

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class UMLImageDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        path, label_id = self.examples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print("🚫 Bad image at runtime, skipping:", path, "|", repr(e))
            new_idx = (idx + 1) % len(self.examples)
            path, label_id = self.examples[new_idx]
            img = Image.open(path).convert("RGB")

        inputs = processor(images=img, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)  # (3, H, W)
        return pixel_values, label_id


train_ds = UMLImageDataset(train_ex)
val_ds   = UMLImageDataset(val_ex)
test_ds  = UMLImageDataset(test_ex)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
import torch
from torch import nn
from transformers import CLIPModel
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

base_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

for p in base_clip.parameters():
    p.requires_grad = False

vision_blocks = base_clip.vision_model.encoder.layers
NUM_UNFREEZE = 12  # last 8 blocks

for blk in vision_blocks[-NUM_UNFREEZE:]:
    for p in blk.parameters():
        p.requires_grad = True

for p in base_clip.visual_projection.parameters():
    p.requires_grad = True

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    task_type="FEATURE_EXTRACTION",
)

base_clip = get_peft_model(base_clip, lora_config)
base_clip.print_trainable_parameters()


class CLIPUMLClassifier(nn.Module):
    def __init__(self, clip_model, num_labels):
        super().__init__()
        self.clip = clip_model
        embed_dim = self.clip.config.projection_dim
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, pixel_values):
        img_feats = self.clip.get_image_features(pixel_values=pixel_values)
        logits = self.classifier(img_feats)
        return logits


model = CLIPUMLClassifier(base_clip, num_labels).to(device)

Device: cuda


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

trainable params: 1,966,080 || all params: 153,243,393 || trainable%: 1.2830


In [ ]:
criterion = nn.CrossEntropyLoss()

trainable_params = list(model.classifier.parameters()) + [
    p for p in model.clip.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(trainable_params, lr=5e-5)

print(
    "Trainable params:",
    sum(p.numel() for p in trainable_params)
)

Trainable params: 1971723


In [ ]:
def run_epoch(model, loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for pixel_values, labels in loader:
        pixel_values = pixel_values.to(device)

        if not isinstance(labels, torch.Tensor):
            labels = torch.tensor(labels, dtype=torch.long, device=device)
        else:
            labels = labels.to(device, dtype=torch.long)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            logits = model(pixel_values)
            loss = criterion(logits, labels)

            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

In [ ]:
EPOCHS = 10
best_val_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, train=True)
    val_loss, val_acc     = run_epoch(model, val_loader,   train=False)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = model.state_dict()

if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_acc = run_epoch(model, test_loader, train=False)
print(f"Test | loss={test_loss:.4f}, acc={test_acc:.3f}")

SAVE_PATH = "uml_clip_lora_classifier_N8.pt"

torch.save(
    {
        "model_state": model.state_dict(),
        "label_to_id": label_to_id,
        "id_to_label": id_to_label,
    },
    SAVE_PATH,
)

print("✅ Saved fine-tuned model to:", SAVE_PATH)

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 01 | train_loss=2.0917, train_acc=0.343 | val_loss=1.6091, val_acc=0.595


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 02 | train_loss=1.1062, train_acc=0.710 | val_loss=0.8774, val_acc=0.782


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

Epoch 08 | train_loss=0.1459, train_acc=0.976 | val_loss=0.3867, val_acc=0.883


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 09 | train_loss=0.1111, train_acc=0.983 | val_loss=0.3694, val_acc=0.895


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 10 | train_loss=0.0842, train_acc=0.988 | val_loss=0.3389, val_acc=0.900
🚫 Bad image at runtime, skipping: /content/drive/MyDrive/clean1/clean/UML State Machine Diagram/img_0325 (2).jpg | UnidentifiedImageError("cannot identify image file '/content/drive/MyDrive/clean1/clean/UML State Machine Diagram/img_0325 (2).jpg'")
Test | loss=0.3904, acc=0.893
✅ Saved fine-tuned model to: uml_clip_lora_classifier_N8.pt
